In [ ]:
import random
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

class manbot():
    def __init__(self, name, coop_stat):
        self.name = name


        self.coop_stat = coop_stat

    def __repr__(self):
        return f"manbot(name={self.name}, coop_stat={self.coop_stat})"


class node():
    def __init__(self, xpos, ypos):
        self.xpos = xpos
        self.ypos = ypos

    def __repr__(self):
        return f"node(xpos={self.xpos:.2f}, ypos={self.ypos:.2f})"


class mapper():
    def __init__(self, repete=20, grid=100):
        self.grid = grid
        self.node_dic = {}

        i = 0
        while i < repete: 
            xx = random.uniform(0, grid)
            yy = random.uniform(0, grid)

            new = node(xx, yy)
            # Ensure unique node coordinates
            if not any(abs(n.xpos - xx) < 1e-5 and abs(n.ypos - yy) < 1e-5 for n in self.node_dic):
                self.node_dic.update({new: {'connections': [], 'attributes': [], "inhabitants": []}})
                i += 1


class connector():
    def __init__(self, node_dic, av=25.0):
        self.nodes = node_dic

        # 1. Create connections based on a threshold radius
        for node1 in self.nodes:
            for node2 in self.nodes:
                if node1 != node2:
                    dist = (((node1.xpos - node2.xpos) ** 2) +
                            ((node1.ypos - node2.ypos) ** 2)) ** 0.5

                    if dist <= av:
                        if node2 not in self.nodes[node1]['connections']:
                            self.nodes[node1]['connections'].append(node2)

        # 2. Guarantee no node remains completely isolated
        for node1 in self.nodes:
            if len(self.nodes[node1]['connections']) == 0:
                closest_node = None
                closest_dist = float('inf')

                for node2 in self.nodes:
                    if node1 != node2:
                        dist = (((node1.xpos - node2.xpos) ** 2) +
                                ((node1.ypos - node2.ypos) ** 2)) ** 0.5

                        if dist < closest_dist:
                            closest_dist = dist
                            closest_node = node2

                if closest_node:
                    self.nodes[node1]['connections'].append(closest_node)

        # 3. Build a graph representation to inspect overall component connection
        G = nx.Graph()
        for n in self.nodes:
            G.add_node(n)
        for n in self.nodes:
            for neighbor in self.nodes[n]['connections']:
                G.add_edge(n, neighbor)

        # 4. Bridge sub-graphs safely using a minimum-spanning approach
        while not nx.is_connected(G):
            components = list(nx.connected_components(G))
            if len(components) <= 1:
                break

            best_pair = None
            best_dist = float('inf')

            for i in range(len(components)):
                for j in range(i + 1, len(components)):
                    comp1 = components[i]
                    comp2 = components[j]

                    for n1 in comp1:
                        for n2 in comp2:
                            dist = (((n1.xpos - n2.xpos) ** 2) +
                                    ((n1.ypos - n2.ypos) ** 2)) ** 0.5

                            if dist < best_dist:
                                best_dist = dist
                                best_pair = (n1, n2)

            if best_pair:
                node1, node2 = best_pair
                self.nodes[node1]['connections'].append(node2)
                self.nodes[node2]['connections'].append(node1)
                G.add_edge(node1, node2)

In [ ]:
class populator():
    def __init__(self, nodee_dic, name_count=1):
        self.nodd = nodee_dic
        self.name_count = name_count
        chance = [1, 2, 3]

        for nodd in self.nodd:
            self.nodd[nodd]['inhabitants'].clear()
                
                

        for nodd in self.nodd:
            i = 0 
            while i < 5:
                bot_name = 'Bot' + str(self.name_count)
                coop_stat = random.choice(chance)
                new_bot = manbot(bot_name, coop_stat)
    
                nodee_dic[nodd]['inhabitants'].append(new_bot)
                self.name_count += 1
                i += 1


class mover():
    def __init__(self, node_dicc, node, bot):
        self.node_dic = node_dicc
        self.node = node
        self.bot = bot
    
        chancer_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        chance = random.choice(chancer_list)
        chance2 = random.choice(chancer_list)

        if chance == 10 and chance2 == (1 or 10):
            self.node_dic[self.node]['inhabitants'].remove(self.bot)

        else:

            destination = random.choice(self.node_dic[self.node]['connections'])

            if (len(self.node_dic[self.node]['inhabitants']) > 10) and (chance%3 == 0):
                    
                self.node_dic[destination]['inhabitants'].append(self.bot)
                if self.bot in self.node_dic[self.node]['inhabitants']:
                    self.node_dic[self.node]['inhabitants'].remove(self.bot)
                
                    
                if (chance2%3 == 0):
                    self.node_dic[destination]['inhabitants'].remove(self.bot)
                        
    
            elif (len(self.node_dic[self.node]['inhabitants']) < 2):
                if random.choice([True, False]):
                    destination = random.choice(self.node_dic[self.node]['connections'])
                    self.node_dic[destination]['inhabitants'].append(self.bot)
                    if self.bot in self.node_dic[self.node]['inhabitants']:
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)
                
        
            elif (chance == 10) and (chance2 != 10):
                    destination = random.choice(self.node_dic[self.node]['connections'])
                    self.node_dic[destination]['inhabitants'].append(self.bot)
                    if self.bot in self.node_dic[self.node]['inhabitants']:
                            self.node_dic[self.node]['inhabitants'].remove(self.bot)
            
            elif (chance == 10) and (chance2 == 10):
                    if self.bot in self.node_dic[self.node]['inhabitants']:
                        self.node_dic[self.node]['inhabitants'].remove(self.bot)


class player():
    def __init__(self, node_dicc, node, name_count):
        self.node_dic = node_dicc
        self.node = node
        self.name_count = name_count
    
        fight_club = list(self.node_dic[node]['inhabitants'])

        # Only play a game if there are at least two bots on the node
        if len(fight_club) > 1:
            bot1 = random.choice(fight_club)
            fight_club.remove(bot1)
            bot2 = random.choice(fight_club)
        
            # Strategy definitions:
            # Strategies 1 and 2 = Cooperation
            # Strategy 3 = Defection

            # SCENARIO 1: Both Cooperate (Coop vs Coop)
            # 50% chance of reproducing a new cooperator, 50% chance nothing happens
            if (bot1.coop_stat in [1,2]) and (bot2.coop_stat in [1,2]):
                if random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2,3,4,5,6]:
                    self.name_count += 1
                    new_bot = manbot('Bot' + str(self.name_count), bot1.coop_stat)
                    self.node_dic[node]['inhabitants'].append(new_bot)
        
            # SCENARIO 2: Bot1 Cooperates, Bot2 Defects (Coop vs Defect)
            # The cooperator (bot1) dies
            elif (bot1.coop_stat in [1,2]) and (bot2.coop_stat == 3):
                
                if random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2,3,4]:
                    self.name_count += 1
                    new_bot = manbot('Bot' + str(self.name_count), bot2.coop_stat)
                    self.node_dic[node]['inhabitants'].append(new_bot)
                    
            # SCENARIO 3: Bot1 Defects, Bot2 Cooperates (Defect vs Coop)
            # The defector (bot1) reproduces another defector
            elif (bot1.coop_stat == 3) and (bot2.coop_stat in [1,2]):
                
                if random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2,3,4,5,6,7,8]:
                    self.name_count += 1
                    new_bot = manbot('Bot' + str(self.name_count), bot1.coop_stat)
                    self.node_dic[node]['inhabitants'].append(new_bot)
        
            # SCENARIO 4: Both Defect (Defect vs Defect)
            # Mutual destruction: both defectors die
            elif (bot1.coop_stat == 3) and (bot2.coop_stat == 3):
                if (bot1 in self.node_dic[node]['inhabitants']) and (bot2 in self.node_dic[node]['inhabitants']):

                    if random.choice([1,2,3,4,5,6,7,8,9,10]) in [1,2]:

                        temp_list = []
                        temp_list.append(bot1)
                        temp_list.append(bot2)
    
                        temu_bot = random.choice(temp_list)
                        
                        self.node_dic[node]['inhabitants'].remove(temu_bot)
                    


def execute_simulation_step(node_dic, current_count):
    """Executes a single step of spatial shifting and player interactions across the dictionary."""
    for nodel in list(node_dic.keys()):
        inhabitants = list(node_dic[nodel]['inhabitants'])
        if len(inhabitants) > 0:
            # Shift bots based on movement rules
            for bott in inhabitants:
                my_mover = mover(node_dic, nodel, bott)
            # Handle game outcomes
            p = player(my_mover.node_dic, nodel, current_count)
            current_count = p.name_count
    return current_count

In [ ]:
my_mapper = mapper(repete=400, grid=20)
my_connector = connector(my_mapper.node_dic, av=1.0)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
def animate_evolving_population_with_strategy(nodess, n_iterations=50, gridsize = 20):

    
    my_populator = populator(nodess)

    #print (my_populator.nodd)
    
    node_dic = nodess
    current_count = my_populator.name_count
    
    # Create figure with extra room on the right for a colorbar legend
    fig, ax = plt.subplots(figsize=(15, 7))
    
    # Pre-build NetworkX layout
    G = nx.Graph()
    pos = {}
    for node_obj in node_dic:
        G.add_node(node_obj)
        pos[node_obj] = (node_obj.xpos, node_obj.ypos)
        for target in node_dic[node_obj]['connections']:
            G.add_edge(node_obj, target)


    #Fix colorbar (added purple)
    custom_colors = ['#d62728', '#9467bd', '#1f77b4']
    rpb_cmap = LinearSegmentedColormap.from_list("RedPurpleBlue", custom_colors)
    
    # Create a persistent colorbar axis so it doesn't duplicate every frame
    cax = fig.add_axes([0.88, 0.15, 0.03, 0.7])
    sm = plt.cm.ScalarMappable(cmap=rpb_cmap, norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Strategy Dominance (Defection vs Cooperation)', rotation=270, labelpad=15, fontweight='bold')
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Defection', '50 / 50 Mix', 'Cooperation'])

    def update(frame):
        nonlocal current_count
        ax.clear()
        
        if frame > 0:
            i = 0
            while i < 10:
                current_count = execute_simulation_step(node_dic, current_count)
                i += 1
            
        # Lists to separate populated vs empty nodes for distinct styling
        live_nodes = []
        live_sizes = []
        live_colors = []
        
        empty_nodes = []
        
        total_bots = 0
        
        # Categorize nodes based on active population
        for node_obj in G.nodes:
            bots = node_dic[node_obj]['inhabitants']
            population_size = len(bots)
            total_bots += population_size
            
            if population_size > 0:
                live_nodes.append(node_obj)
                live_sizes.append(50 + population_size * 12)
                
                # Strategy ratio tracking
                cooperators = sum(1 for b in bots if b.coop_stat in [1, 2])
                coop_ratio = cooperators / population_size
                live_colors.append(coop_ratio)
            else:
                empty_nodes.append(node_obj)
                
        # 1. Draw standard background connection edges
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gainsboro', width=1.2)
        
        # 2. Draw live nodes with filled strategy colors (Red to Blue)
        if live_nodes:
            nx.draw_networkx_nodes(
                G, pos, ax=ax, nodelist=live_nodes, node_size=live_sizes, 
                node_color=live_colors, cmap=plt.cm.RdBu, vmin=0, vmax=1
            )
            
        # 3. Draw empty nodes as completely hollow, thin outlines
        if empty_nodes:
            nx.draw_networkx_nodes(
                G, pos, ax=ax, nodelist=empty_nodes, 
                node_size=40,            # Small uniform baseline size
                node_color='none',        # Transparent/hollow center
                edgecolors='dimgray',     # Clear gray border ring
                linewidths=1.5,           # Noticeable border thickness
                node_shape='o'            # Standard circle shape
            )
            
            # Optional: To make the hollow circles dashed, we grab the 
            # last drawn collection on the axis and apply a dash style
            ax.collections[-1].set_linestyle((0, (3, 3)))
        
        # Consistent layout limits
        padding = gridsize * 0.05
        ax.set_xlim(-padding, gridsize + padding)
        ax.set_ylim(-padding, gridsize + padding)
        ax.set_title(f"Iteration: {frame*10}  |  Active Population: {total_bots}", fontsize=12, fontweight='bold')
        ax.axis('off')

    ani = FuncAnimation(fig, update, frames=(n_iterations//10), repeat=False)
    plt.close(fig)
    return ani

In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 125.0



strategy_animation = animate_evolving_population_with_strategy(my_mapper.node_dic, n_iterations=3500, gridsize = 20)
HTML(strategy_animation.to_jshtml())



In [ ]:
strategy_animation.save("network_evolution4.gif", writer = "pillow", fps = 5)